<a href="https://colab.research.google.com/github/huyle2411-hub/credit-risk-scorecard/blob/main/notebooks/02_cleaning_woe_iv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Credit Scorecard — Notebook 02: Cleaning + WOE/IV

**Mục tiêu notebook này:** Làm sạch dữ liệu theo các quyết định từ 01, rồi tính WOE/IV để chọn biến cho scorecard.


In [ ]:
!pip install optbinning -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.8/214.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 23.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 5.26.1 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 5.26.1 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 5.26.1 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
from optbinning import BinningProcess

pd.set_option('display.max_columns', None)

## 1. Nạp lại dữ liệu

Notebook mới nên phải upload lại `cs-training.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('cs-training.csv', index_col=0)
target = 'SeriousDlqin2yrs'
print('Nap:', df.shape)

Saving cs-training.csv to cs-training.csv
Nap: (150000, 11)


##2. Cleaning
**Thực hiện 5 quyết định làm sạch dữ liệu:** (1) đánh dấu các giá trị 96/98 và chuyển về NaN, (2) chuyển `age = 0` thành NaN, (3) giới hạn `Utilization` và `DebtRatio` tại phân vị 99% để giảm ảnh hưởng của outlier, (4) thay giá trị thiếu của `NumberOfDependents` bằng 0, (5) giữ nguyên NaN của `MonthlyIncome` để WOE tạo một bin riêng cho nhóm thiếu.

In [ ]:
d = df.copy()
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse',
             'NumberOfTimes90DaysLate',
             'NumberOfTime60-89DaysPastDueNotWorse']

# (1) danh dau nhom bat thuong, roi dua 96/98 ve NaN de khong lam meo bins
d['severe_delinq_flag'] = (d[late_cols] >= 90).any(axis=1).astype(int)
d[late_cols] = d[late_cols].mask(d[late_cols] >= 90)

# (2) age = 0 la vo ly
d['age'] = d['age'].replace(0, np.nan)

# (3) cap outlier o phan vi 99% (winsorize)
for c in ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']:
    cap = d[c].quantile(0.99)
    d[c] = d[c].clip(upper=cap)

# (4) so nguoi phu thuoc thieu -> 0
d['NumberOfDependents'] = d['NumberOfDependents'].fillna(0)

# (5) MonthlyIncome: giu NaN, de WOE tao bin "Missing" rieng (missing co the du bao)

print('Sau lam sach:', d.shape, '| severe_delinq_flag =1:', int(d['severe_delinq_flag'].sum()))
d.head()

Sau lam sach: (150000, 12) | severe_delinq_flag =1: 269


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,severe_delinq_flag
1,1,0.766127,45.0,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0,0
2,0,0.957151,40.0,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0,0
3,0,0.658180,38.0,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0,0
4,0,0.233810,30.0,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0,0
5,0,0.907239,49.0,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0,0


##3. WOE/IV
WOE mã hóa từng khoảng giá trị theo tỷ lệ khách hàng tốt và xấu, giúp Logistic Regression khai thác hiệu quả các quan hệ phi tuyến và nhóm giá trị thiếu. IV phản ánh sức mạnh dự báo của từng biến, dùng để xếp hạng và chọn feature.

In [ ]:
X = d.drop(columns=[target])
y = d[target].values

bp = BinningProcess(variable_names=list(X.columns))
bp.fit(X.values, y)

iv_table = bp.summary().sort_values('iv', ascending=False)[['name', 'iv']].reset_index(drop=True)
iv_table

,name,iv
0,RevolvingUtilizationOfUnsecuredLines,1.112014
1,NumberOfTimes90DaysLate,0.839456
2,NumberOfTime30-59DaysPastDueNotWorse,0.745862
3,age,0.264253
4,NumberOfOpenCreditLinesAndLoans,0.084626
5,MonthlyIncome,0.080132
6,DebtRatio,0.077609
7,NumberRealEstateLoansOrLines,0.055354
8,NumberOfTime60-89DaysPastDueNotWorse,0.039117
9,NumberOfDependents,0.033818


Ba biến có IV cao nhất là `RevolvingUtilizationOfUnsecuredLines` (khoảng 1.11) và hai biến DPD (khoảng 0.75 đến 0.84), đóng vai trò là các feature quan trọng nhất. IV lớn hơn 0.5 ở đây không phải do leakage mà phản ánh đúng bản chất của dữ liệu tín dụng. `severe_delinq_flag` chỉ có IV khoảng 0.04 vì tín hiệu đã được các biến DPD nắm bắt, nên được giữ như một feature bổ sung thay vì feature chính.

In [7]:
keep = iv_table[iv_table['iv'] >= 0.05]['name'].tolist()
drop = iv_table[iv_table['iv'] < 0.05]['name'].tolist()
print('GIU (%d):' % len(keep), keep)
print('LOAI (%d):' % len(drop), drop)

GIU (8): ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'age', 'NumberOfOpenCreditLinesAndLoans', 'MonthlyIncome', 'DebtRatio', 'NumberRealEstateLoansOrLines']
LOAI (3): ['NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'severe_delinq_flag']


**→** dùng danh sách `keep` và phép biến đổi WOE để huấn luyện logistic scorecard, đánh giá AUC/KS/Gini, và so với 1 boosting challenger.